In [1]:
import os
import shutil
import csv
import PyPDF2
import openai
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ----------------------------------------
# Configuration
# ----------------------------------------
INPUT_FOLDER = "cot_papers/"
OUTPUT_FOLDER = "cot_papers_selected"
CSV_PATH = "cot_papers_selected.csv"
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
MODEL_NAME = "gpt-4o"  # Keep this model, it's faster and cheaper than 4.1-2025-04-14 for classification

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
openai.api_key = OPENAI_API_KEY

# ----------------------------------------
# Prompt Template
# ----------------------------------------
PROMPT_TEMPLATE = """
You are an expert research assistant. Your task is to read the abstract (or the first few paragraphs of the paper's content if an abstract section is not explicitly marked) and determine whether the paper is focused on reducing redundancy in chain-of-thought (CoT) reasoning and making CoT more computationally efficient. If the paper is indeed related to improving reasoning efficiency (e.g., introducing benchmarks like Think-Bench, metrics for overthinking, methods to cut down redundant tokens, or novel activation‐steering approaches such as RASPID), respond with EXACTLY three lines, each separated by a newline:

1. A single-word answer: “Yes” if the paper is about CoT efficiency/redundancy reduction, otherwise “No”.
2. A one‐sentence review of why it is or isn't related.
3. A concise label or keyword list summarizing the core topic.

Now evaluate the following text. Only output the three lines as specified above.
"""

# ----------------------------------------
# PDF Extractor
# ----------------------------------------
def extract_text_from_pdf(pdf_path, max_chars=10000):
    reader = PyPDF2.PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        try:
            full_text += page.extract_text() + "\n"
        except:
            continue

    lowered = full_text.lower()
    if "abstract" in lowered:
        idx = lowered.index("abstract")
        return full_text[idx: idx + max_chars]
    else:
        return full_text[:max_chars]

# ----------------------------------------
# OpenAI Call
# ----------------------------------------
def analyze_paper(text_snippet):
    prompt = PROMPT_TEMPLATE + "\n" + text_snippet.strip() + "\n---\nAnswer:\n"
    try:
        response = openai.ChatCompletion.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are a helpful research assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=150,
        )
        output = response.choices[0].message.content.strip().split("\n")
        if len(output) >= 3:
            yes_no, review, labels = output[0].strip(), output[1].strip(), output[2].strip()
            is_relevant = yes_no.lower() == "yes"
            return is_relevant, review, labels
        else:
            return False, "FormatError", ""
    except Exception as e:
        return False, f"APIError: {str(e)}", ""

# ----------------------------------------
# Worker Function
# ----------------------------------------
def process_file(filename):
    pdf_path = os.path.join(INPUT_FOLDER, filename)
    try:
        reader = PyPDF2.PdfReader(pdf_path)
        title = reader.metadata.title if reader.metadata and reader.metadata.title else filename
    except:
        title = filename

    snippet = extract_text_from_pdf(pdf_path, max_chars=2000)
    is_relevant, one_line_review, labels = analyze_paper(snippet)

    if is_relevant:
        dest_path = os.path.join(OUTPUT_FOLDER, filename)
        shutil.move(pdf_path, dest_path)

    return {
        "title": title,
        "filename": filename,
        "is_relevant": is_relevant,
        "one_line_review": one_line_review,
        "labels": labels
    }

# ----------------------------------------
# Parallel Execution
# ----------------------------------------
if __name__ == "__main__":
    files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(".pdf")]
    results = []

    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {executor.submit(process_file, file): file for file in files}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing PDFs"):
            result = future.result()
            results.append(result)

    df = pd.DataFrame(results)
    df.to_csv(CSV_PATH, index=False, quoting=csv.QUOTE_MINIMAL)
    print(f"Done. Results saved to '{CSV_PATH}'. Relevant papers moved to '{OUTPUT_FOLDER}'.")


Processing PDFs:   3%|█████▍                                                                                                                                                     | 40/1147 [00:20<05:12,  3.55it/s]FloatObject (b'0.0000000-7450581') invalid; use 0.0 instead
FloatObject (b'0.0000000-7450581') invalid; use 0.0 instead
FloatObject (b'0.0000000-7450581') invalid; use 0.0 instead
FloatObject (b'0.0000000-7450581') invalid; use 0.0 instead
Processing PDFs:  67%|██████████████████████████████████████████████████████████████████████████████████████████████████████▌                                                   | 764/1147 [11:38<04:19,  1.48it/s]FloatObject (b'0.00-37975883') invalid; use 0.0 instead
FloatObject (b'0.000-18372704') invalid; use 0.0 instead
Processing PDFs:  91%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊              | 1041/1147 [16:45<01:27,  1.22it/s]unknown widths

Done. Results saved to 'cot_papers_selected.csv'. Relevant papers moved to 'cot_papers_selected'.
